# Module 03 — Training a Network

This notebook is about exactly one thing: **how a network learns**. Module 01 built the autograd engine; Module 02 used it to build a network's structure (`Neuron` → `Layer` → `MLP`), but with random, meaningless weights. This notebook connects the two: define a **loss** (a single `Value` measuring how wrong the network currently is), call `.backward()` to get its gradient with respect to every parameter, and repeatedly nudge each parameter opposite its gradient. That loop — forward pass, loss, backward pass, update — *is* training, for any neural network at any scale.

In [ ]:
# Value, Neuron, Layer, MLP — same classes as Modules 01 and 02, copied in so
# this notebook runs standalone. Not re-explained here; see those modules.
import math
import random

import matplotlib.pyplot as plt
import numpy as np


class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f"**{power}")

        def _backward():
            self.grad += (power * self.data ** (power - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"


class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [
            Layer(sizes[i], sizes[i + 1], nonlin=(i != len(nouts) - 1))
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## A toy problem

Two blobs of 2D points, one labeled `+1`, one labeled `-1`. The task: learn a decision boundary that separates them. Small and visualizable, so you can actually see what the network learned at the end.

In [ ]:
random.seed(42)


def make_blobs(n_per_class=50):
    X, y = [], []
    centers = [(-1.5, -1.5), (1.5, 1.5)]
    for label, (cx, cy) in enumerate(centers):
        for _ in range(n_per_class):
            X.append([cx + random.gauss(0, 0.6), cy + random.gauss(0, 0.6)])
            y.append(1.0 if label == 0 else -1.0)
    return X, y


Xs, ys = make_blobs()

plt.figure(figsize=(5, 5))
plt.scatter([p[0] for p in Xs], [p[1] for p in Xs], c=ys, cmap=plt.cm.Spectral, edgecolors="k")
plt.title("Toy dataset: two classes")
plt.show()

## Defining "wrong": the loss function

Before we can improve the network, we need a single number that says how badly it's doing right now — smaller is better. We'll use two pieces added together:

- **Max-margin (hinge) loss**, per example: `relu(1 - y * score)`. If the network's `score` has the same sign as the true label `y` *and* is confidently large, `1 - y*score` goes negative and `relu` clamps it to 0 — no penalty. If the network is wrong, or right but not confident, this grows — pushing the network toward not just correct, but confidently correct.
- **L2 regularization**: a small penalty (`alpha * sum(p*p for p in parameters)`) on the size of the weights. This discourages the network from relying on any single huge weight, which tends to produce smoother, more sensible decision boundaries.

The crucial thing: `total_loss` is a `Value`, built by running every example through the `model` and combining the results — meaning it's a computation graph that touches *every parameter* in the network. Calling `.backward()` on it gives us the gradient of the loss with respect to every single weight and bias, all at once.

In [ ]:
model = MLP(2, [8, 8, 1])
print(f"model has {len(model.parameters())} parameters")


def compute_loss():
    inputs = [[Value(x) for x in row] for row in Xs]
    scores = [model(x) for x in inputs]

    losses = [(1 + -yi * scorei).relu() for yi, scorei in zip(ys, scores)]
    data_loss = sum(losses) * (1.0 / len(losses))

    alpha = 1e-4
    reg_loss = alpha * sum((p * p for p in model.parameters()))
    total_loss = data_loss + reg_loss

    accuracy = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(ys, scores)]
    return total_loss, sum(accuracy) / len(accuracy)


initial_loss, initial_acc = compute_loss()
print(f"before any training: loss {initial_loss.data:.4f}, accuracy {initial_acc * 100:.1f}%")

## Gradient descent

Each step:
1. Zero out every parameter's `.grad` — gradients accumulate (`+=`) across `.backward()` calls, so leftover values from the previous step would corrupt this one.
2. Recompute the loss (forward pass) and call `.backward()` (backward pass) to fill in fresh gradients.
3. Nudge every parameter a small step *opposite* its gradient: `p.data -= learning_rate * p.grad`. The gradient points in the direction the loss *increases* fastest, so moving against it decreases the loss.

The `learning_rate` controls how big each nudge is. Too large and training can overshoot and diverge; too small and it takes forever. Here it starts at 1.0 and decays toward 0.1 over training — larger steps early while the network is very wrong, smaller steps later to settle in precisely.

In [ ]:
loss_history = []
for step in range(100):
    total_loss, acc = compute_loss()

    for p in model.parameters():
        p.grad = 0.0
    total_loss.backward()

    learning_rate = 1.0 - 0.9 * step / 100
    for p in model.parameters():
        p.data -= learning_rate * p.grad

    loss_history.append(total_loss.data)
    if step % 10 == 0:
        print(f"step {step:3d}   loss {total_loss.data:.4f}   accuracy {acc * 100:.1f}%")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training loss over time")
plt.show()

## Seeing what it learned

Score every point on a grid covering the input space, and shade by which class the network predicts there. If training worked, the shaded regions should roughly match the two blobs from earlier.

In [ ]:
h = 0.25
x_min, x_max = -3.5, 3.5
y_min, y_max = -3.5, 3.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

grid_inputs = [[Value(px), Value(py)] for px, py in zip(xx.ravel(), yy.ravel())]
grid_scores = [model(inp) for inp in grid_inputs]
Z = np.array([s.data > 0 for s in grid_scores]).reshape(xx.shape)

plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.4)
plt.scatter([p[0] for p in Xs], [p[1] for p in Xs], c=ys, cmap=plt.cm.Spectral, edgecolors="k")
plt.title("Decision boundary learned by the network")
plt.show()

## What just happened

Every piece that matters for training *any* neural network — including the LLM we're building toward — appeared in this notebook: a **loss function** turning "how wrong is the model" into one `Value`, a **backward pass** computing its gradient with respect to every parameter, and **gradient descent** using those gradients to make the model less wrong, repeated until it converges.

GPT-scale models do exactly this — just with tensors instead of scalars (millions of numbers processed at once via matrix multiplication), on a GPU, with a fancier optimizer than plain gradient descent (AdamW, covered later). The math doesn't change; only the scale and hardware do.

**Next: Module 04** — the same three ideas (autograd, network structure, training loop), rebuilt with real PyTorch (`torch.Tensor`, `.backward()`, `nn.Module`, `torch.optim`), so you can see the direct correspondence between what you hand-wrote across Modules 01–03 and what the framework does for you.